# Google Capstone Project

# Importing Gemini API Key

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv 

load_dotenv() 
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    raise ValueError("API key not found. Please set the GOOGLE_API_KEY in your .env file.")


genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API Key loaded and configured successfully.")


Gemini API Key loaded and configured successfully.


# Using RAG in my documents


In [ ]:
import os
import google.generativeai as genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.generativeai import types # Keeping 'types' import as it might be useful later (for myself)
from dotenv import load_dotenv
from IPython.display import Markdown, display # For displaying markdown

# --- API Key Setup ---
load_dotenv()
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    raise ValueError("API key not found. Please set the GOOGLE_API_KEY in your .env file.")
genai.configure(api_key=GOOGLE_API_KEY)
print("Gemini API Key loaded and configured successfully.")

# --- Load Documents from TXT Files ---
document_files = [
    "Jan2025.txt",
    "Jun2024.txt",
    "March2025.txt",
    "WeeklyWorkout.txt"
]
documents = []
doc_ids = []

print(f"\nAttempting to load {len(document_files)} documents from .txt files...")
for i, filename in enumerate(document_files):
    doc_id = f"doc_{i+1}"
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
            documents.append(content)
            doc_ids.append(doc_id)
            print(f"Successfully loaded '{filename}' (ID: {doc_id})")
    except FileNotFoundError:
        print(f"ERROR: File not found - '{filename}'.")
    except Exception as e:
        print(f"ERROR: Could not read file '{filename}'. Reason: {e}")

if len(documents) == len(document_files):
    print(f"\nSuccessfully loaded content for {len(documents)} documents.")
else:
     print(f"\nWarning: Only loaded {len(documents)} out of {len(document_files)} documents.")

# --- Gemini Embedding Function for ChromaDB ---
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_name="models/text-embedding-004", task_type="retrieval_document"):
        self.model_name = model_name
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        response = genai.embed_content(
            model=self.model_name,
            content=input,
            task_type=self.task_type
        )
        return response['embedding']

# --- Setup ChromaDB & Add Documents ---
print("\n--- Setting up ChromaDB ---")
chroma_client = chromadb.Client()
db_collection_name = "fitness_logs"
embed_fn_docs = GeminiEmbeddingFunction(task_type="retrieval_document")

db = chroma_client.get_or_create_collection(
    name=db_collection_name,
    embedding_function=embed_fn_docs
)

try:
    db.upsert(documents=documents, ids=doc_ids)
    print(f"Successfully added/updated {len(documents)} documents.")
    print(f"Total documents in collection: {db.count()}")
except Exception as e:
    print(f"Error adding/updating documents: {e}")
    print(f"Document count remains: {db.count()}")

# --- RAG Example ---
print("\n--- RAG Example ---")
user_query_rag = "How many times did I run 10km the last year?"
print(f"User Query: {user_query_rag}")

embed_fn_query = GeminiEmbeddingFunction(task_type="retrieval_query")

query_results = db.query(
    query_texts=[user_query_rag],
    n_results=3,
    include=['documents']
)

retrieved_docs = query_results.get('documents', [[]])[0]
if not retrieved_docs:
    print("Could not retrieve relevant documents for the query.")
    rag_context = "No relevant documents found."
else:
    rag_context = "\n\n---\n\n".join(retrieved_docs)
    print(f"\nRetrieved Context (Top {len(retrieved_docs)} chunks):")
    print(f"{rag_context[:500]}...") # Show beginning of context for reference

rag_prompt = f"""You are a helpful fitness assistant analyzing workout logs. Answer the user's question based *only* on the provided context. If the context doesn't contain the answer, say so clearly.

Context from workout logs:
---
{rag_context}
---

User Question: {user_query_rag}

Answer:"""

print("\nGenerating RAG Answer...")
try:
    rag_model = genai.GenerativeModel('gemini-1.5-flash')
    rag_response = rag_model.generate_content(rag_prompt)
    print("\nGenerated RAG Answer:")
    # Use display(Markdown(...)) for formatted output in Jupyter/VS Code notebooks
    display(Markdown(rag_response.text))
except Exception as e:
    print(f"Error generating RAG response: {e}")

print("\n--- RAG Section Finished ---")

Gemini API Key loaded and configured successfully.

Attempting to load 4 documents from .txt files...
Successfully loaded 'Jan2025.txt' (ID: doc_1)
Successfully loaded 'Jun2024.txt' (ID: doc_2)
Successfully loaded 'March2025.txt' (ID: doc_3)
Successfully loaded 'WeeklyWorkout.txt' (ID: doc_4)

Successfully loaded content for 4 documents.

--- Setting up ChromaDB ---
Successfully added/updated 4 documents.
Total documents in collection: 4

--- RAG Example ---
User Query: How many times did I run 10km the last year?

Retrieved Context (Top 3 chunks):
Log workout 2
25-mar-2025

Chest 
Bike zone 2


Oatmeal cookies
Salmon 
Etc

26-mar-2025

Back
Cardio

27-mar-2025

Oatmeal + pineapple
3 boiled eggs
Some avocado


28-march-2024

Leg workout

Oatmeal paw paw

3 egg whites
Ham slices
Avocado
1 small donut
Cappuccino decaf

29-mar-2025

Shoulders
Jumping rope and boxing

76kgs

30-mar-2025

10 km run

31-mar-2025

Chest

Meals
Oatmeal cookies, protein shake with banana, three boiled eggs avoc

Based on the provided logs, you ran 10km four times in the last year.



--- RAG Section Finished ---


In [ ]:
# --------------------------------------------------------------------------
# 7. STRUCTURED OUTPUT (JSON MODE) EXAMPLE (All Outputs)
# --------------------------------------------------------------------------
import json
from IPython.display import display, Markdown

print("\n--- Structured Output (JSON) Example ---")

target_doc_index = 1 # Index 1 corresponds to "Jun2024.txt"

if target_doc_index < len(documents):
    target_doc_content = documents[target_doc_index]
    target_doc_name = document_files[target_doc_index]
    target_date = "19-August-2024"

    print(f"Target Document: '{target_doc_name}'")
    print(f"Target Date for Extraction: {target_date}")

    # --- Prepare Prompt for JSON Output ---
    json_prompt = f"""
Analyze the document content from '{target_doc_name}' below. Find the entry for "{target_date}".
Extract the workout activities and meals for that day.
Return ONLY a valid JSON object with keys: "date", "workout_summary", "meals" (list of strings).
If the date isn't found or info is missing for that date, use null or empty values in the JSON structure, but still return a valid JSON object.

Document Content from '{target_doc_name}':
---
{target_doc_content}
---

JSON Output:
"""

    # --- Configure for JSON Output ---
    json_generation_config = genai.types.GenerationConfig(
        response_mime_type='application/json',
    )

    # --- Generate JSON using Gemini ---
    print("\nGenerating Structured (JSON) Output...")
    try:
        json_model = genai.GenerativeModel('gemini-1.5-flash')
        json_response = json_model.generate_content(
            contents=json_prompt,
            generation_config=json_generation_config,
        )

        # --- Print the Raw JSON Response ---
        print("\nRaw JSON Response Text from Model:")
        print(json_response.text) # Output 1: Raw JSON

        # --- Parse and Display Formatted Output ---
        print("\nProcessing and Formatting Output...")
        try:
            json_text = json_response.text
            parsed_json = json.loads(json_text)

       
            print("\nParsed JSON:")
            print(json.dumps(parsed_json, indent=2)) 

            # Extract data for Markdown formatting
            entry_date = parsed_json.get("date", "N/A")
            workout_summary = parsed_json.get("workout_summary", "No workout information found.")
            meals_list = parsed_json.get("meals", [])

            # --- Build & Display Markdown Output ---
            md_output = f"### Extraction Results for {target_date} from '{target_doc_name}'\n\n"
            md_output += f"**(Extracted Date:** {entry_date})\n\n"
            md_output += f"**Workout:**\n{workout_summary}\n\n"
            md_output += "**Meals:**\n"
            if meals_list:
                for meal in meals_list:
                    md_output += f"- {meal}\n"
            else:
                md_output += "- No meal information found.\n"

            print("\nFormatted Output (Markdown):")
            display(Markdown(md_output)) # Output 3: Custom Markdown

        except json.JSONDecodeError:
            print("\nERROR: Could not parse the model's output as valid JSON.")
        except Exception as display_e:
            print(f"\nERROR: An error occurred during output formatting: {display_e}")

    except Exception as e:
        print(f"Error generating JSON response: {e}")

else:
    print(f"ERROR: Cannot run Section 7. Document index {target_doc_index} is out of bounds.")
    print("Please ensure the document files were loaded correctly in Section 3.")


print("\n--- Structured Output Section Finished ---")


--- Structured Output (JSON) Example ---
Target Document: 'Jun2024.txt'
Target Date for Extraction: 19-August-2024

Generating Structured (JSON) Output...

Raw JSON Response Text from Model:
{"date": "19-August-2024", "workout_summary": "Chest workout Swimming workout at night", "meals": ["Oatmeal w paw paw", "3 boiled white eggs w 2 turkey slices w panela cheese", "Chicken w Rice", "Apple", "Chicken w Rice", "Protein shake"]}

Processing and Formatting Output...

Parsed and Pretty-Printed JSON:
{
  "date": "19-August-2024",
  "workout_summary": "Chest workout Swimming workout at night",
  "meals": [
    "Oatmeal w paw paw",
    "3 boiled white eggs w 2 turkey slices w panela cheese",
    "Chicken w Rice",
    "Apple",
    "Chicken w Rice",
    "Protein shake"
  ]
}

Formatted Output (Markdown):


### Extraction Results for 19-August-2024 from 'Jun2024.txt'

**(Extracted Date:** 19-August-2024)

**Workout:**
Chest workout Swimming workout at night

**Meals:**
- Oatmeal w paw paw
- 3 boiled white eggs w 2 turkey slices w panela cheese
- Chicken w Rice
- Apple
- Chicken w Rice
- Protein shake



--- Structured Output Section Finished ---
